# Week 1-1 · 요청 모델을 직접 정의하고 검증하기

## 시나리오
HR 규정 질문 API가 검색을 시작하기 전에 빈 질문과 지나치게 짧은 질문을 거부하도록 입력 계약을 만듭니다.

## 학습 목표
- Pydantic `BaseModel`과 `Field`로 입력 경계를 선언한다.
- 정상 입력과 `ValidationError`의 구조를 비교한다.
- 검증이 RAG pipeline보다 먼저 필요한 이유를 설명한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · 모델 선언

In [ ]:
# 실행 순서: 1단계 · 모델 선언에서 PracticeQuery, PracticeResponse을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · 모델 선언.
from pydantic import BaseModel, Field, ValidationError

# API 요청이 retrieval에 들어가기 전에 질문 길이를 검증하는 입력 경계입니다.
class PracticeQuery(BaseModel):
    question: str = Field(min_length=3, max_length=120)

# 검증된 질문과 처리 상태를 호출자에게 돌려주는 최소 응답 계약입니다.
class PracticeResponse(BaseModel):
    status: str
    normalized_question: str

practice_question = "휴가는 며칠 전에 신청해야 하나요?"
valid_query = PracticeQuery(question=practice_question)
valid_query.model_dump()

### 2단계 · 실패 구조 관찰

In [ ]:
# 실행 순서: 2단계 · 실패 구조 관찰에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · 실패 구조 관찰.
try:
    PracticeQuery(question="x")
except ValidationError as error:
    practice_error = error.errors()[0]

practice_error

### 3단계 · 입력 경계 계약

In [ ]:
# 실행 순서: 3단계 · 입력 경계 계약에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · 입력 경계 계약.
practice_response = PracticeResponse(status="accepted", normalized_question=valid_query.question.strip())
assert valid_query.question == practice_question
assert practice_response.status == "accepted"
assert practice_error["type"] == "string_too_short"
assert practice_error["loc"] == ("question",)
{"request": valid_query.model_dump(), "response": practice_response.model_dump(), "rejected": practice_error["type"]}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
검증 실패 입력은 retriever나 graph에 전달하지 않습니다. 오류를 기본 질문으로 몰래 바꾸지 않습니다.

## 실제 app 연결
Week 1 app의 요청 모델도 같은 방식으로 API 입력을 검증한 뒤 질문만 graph에 전달합니다. 여기서는 app 모델을 import하지 않고 동일한 경계 원리를 작은 모델로 재현했습니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `02_documents_and_splitting.ipynb`에서는 검증된 질문이 검색할 정책 원문을 `Document`와 chunk로 구성합니다.